# Foundation Model Embedding Extraction
## From Gene Sequences to Per-Nucleotide Embeddings

This notebook walks through extracting foundation model embeddings for gene sequences,
using either **Evo2** (CUDA) or **HyenaDNA** (MPS/CPU/CUDA).

1. **HuggingFace Setup** — Authentication for gated model access
2. **Resource Check** — Can your hardware run Evo2?
3. **Load Gene Sequences** — Via the main data pipeline (GTF + FASTA)
4. **Load Foundation Model** — Evo2 on CUDA, or HyenaDNA on MPS/CPU
5. **Extract Embeddings** — Chunk, encode, stitch per-nucleotide vectors
6. **Build Exon Labels** — Binary labels from splice site annotations
7. **Inspect & Visualize** — Verify outputs, explore embedding space

### Model Backends

| Model | Device | Context | Params | Notes |
|-------|--------|---------|--------|-------|
| **Evo2** 7B/40B | CUDA only | 1M bp | 7B/40B | Official `evo2` package |
| **HyenaDNA** | MPS, CPU, CUDA | up to 1M bp | ~54M | HuggingFace `transformers` |

See `examples/foundation_models/docs/foundation-model-catalog.md` for the full model catalog
(SpliceBERT, AlphaGenome, Pangolin, DNABERT-2, etc.).

### Prerequisites
```bash
pip install -e .                    # main project
pip install -e ./foundation_models  # foundation models sub-project
```

### Related
- `examples/foundation_models/02_embedding_extraction.py` — CLI version (supports `--model evo2|hyenadna`)
- `notebooks/foundation_models/01_training_pipeline.ipynb` — Synthetic training (no foundation model needed)
- `examples/foundation_models/01_synthetic_pipeline.py` — End-to-end with synthetic data

## Step 0: HuggingFace Authentication

Evo2 (`arcinstitute/evo2-7b`) is a **gated model** on HuggingFace. You need to:

1. **Create an account** at [huggingface.co](https://huggingface.co/join)
2. **Accept the license** at [arcinstitute/evo2-7b](https://huggingface.co/arcinstitute/evo2-7b)
3. **Create an access token** at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (read access is sufficient)
4. **Authenticate** using one of the methods below

This is a **one-time setup** per machine — the token is saved to `~/.cache/huggingface/`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads .env into os.environ (picks up HF_TOKEN)

# --- Option A: Login interactively (if no .env / env var set) ---
# Uncomment and run once. It will prompt you to paste your token.
# from huggingface_hub import login
# login()

# --- Option B: Set token directly (e.g., from a secrets manager) ---
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# --- Verify authentication ---
from huggingface_hub import HfApi

try:
    api = HfApi()
    user = api.whoami()
    print(f"Authenticated as: {user['name']}")
except Exception as e:
    print(f"Not authenticated: {e}")
    print()
    print("To fix, either:")
    print("  1. Add HF_TOKEN=hf_... to your .env file")
    print("  2. Uncomment Option A above and run this cell again")
    print("  3. Run in terminal: huggingface-cli login")

## Step 1: Resource Check

Quick check of your hardware. Evo2 requires CUDA + ~7 GB VRAM; HyenaDNA runs on any device.

In [ ]:
import torch

from foundation_models.utils.resources import (
    check_current_hardware,
    estimate_embedding_extraction,
    print_feasibility_report,
)

hw = check_current_hardware()
has_cuda = torch.cuda.is_available()
print(f"Hardware: {hw['label']} ({hw['device']}, {hw['memory_gb']:.1f} GB)")
print(f"CUDA available: {has_cuda}")
print()

if has_cuda:
    MODEL_SIZE = "7b"
    result = estimate_embedding_extraction(model_size=MODEL_SIZE, n_genes=2)
    if result["feasible"]:
        print(f"Evo2 {MODEL_SIZE}: {result['notes'][0]}")
    else:
        print(f"Evo2 {MODEL_SIZE} may not fit. Consider HyenaDNA instead.")
        for note in result["notes"]:
            print(f"  {note}")
    print(f"\nDefault backend: Evo2 (CUDA detected)")
else:
    print("No CUDA — will use HyenaDNA (works on MPS/CPU).")
    print("HyenaDNA medium-160k needs ~1 GB RAM. You have plenty.")

## Step 2: Load Gene Sequences

We use `prepare_gene_data()` from the main pipeline. This handles:
- GTF annotation parsing (MANE reference)
- FASTA sequence extraction
- Gene symbol → genomic coordinates resolution

The data lives in `data/mane/GRCh38/` (FASTA + GTF symlinks).

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from agentic_spliceai.splice_engine.base_layer.data.preparation import prepare_gene_data

# Choose genes to extract embeddings for
GENES = ["BRCA1", "TP53"]

gene_data = prepare_gene_data(
    genes=GENES,
    build="GRCh38",
    annotation_source="mane",
)

print(f"\nLoaded {len(gene_data)} genes:")
for row in gene_data.iter_rows(named=True):
    seq_len = len(row["sequence"]) if row["sequence"] else 0
    print(f"  {row['gene_name']}: {row['seqname']}:{row['start']:,}-{row['end']:,} "
          f"({row['strand']}), {seq_len:,} bp")

## Step 3: Load Foundation Model

**Evo2** requires CUDA — the first load downloads weights (~7 GB for 7B INT8).
**HyenaDNA** works on any device (MPS, CPU, CUDA) — downloads are much smaller (~200 MB).

The cell below auto-detects your hardware and loads the appropriate model.
Set `BACKEND = "evo2"` or `BACKEND = "hyenadna"` to override.

In [ ]:
import time
import torch

# Set to "evo2", "hyenadna", or "auto" (auto-detects CUDA)
BACKEND = "auto"

if BACKEND == "auto":
    BACKEND = "evo2" if torch.cuda.is_available() else "hyenadna"
    print(f"Auto-detected backend: {BACKEND}")

if BACKEND == "evo2":
    from foundation_models.evo2 import Evo2Embedder

    print(f"Loading Evo2 {MODEL_SIZE}...")
    t0 = time.time()
    embedder = Evo2Embedder(model_size=MODEL_SIZE)
    load_time = time.time() - t0

    print(f"  Loaded in {load_time:.1f}s")
    print(f"  Hidden dim: {embedder.model.hidden_dim}")
    print(f"  Device: {embedder.model.device}")

else:
    from foundation_models.hyenadna import HyenaDNAEmbedder

    HYENADNA_SIZE = "medium-160k"  # tiny-1k, small-32k, medium-160k, large-1m
    print(f"Loading HyenaDNA {HYENADNA_SIZE}...")
    t0 = time.time()
    embedder = HyenaDNAEmbedder(model_size=HYENADNA_SIZE)
    load_time = time.time() - t0

    print(f"  Loaded in {load_time:.1f}s")
    print(f"  Hidden dim: {embedder.model.hidden_dim}")
    print(f"  Device: {embedder.model.device}")

## Step 4: Extract Embeddings

For each gene, we:
1. **Chunk** the sequence into overlapping windows (default 8192 bp, 256 overlap)
2. **Encode** each chunk through the foundation model → `[chunk_len, hidden_dim]` tensor
3. **Stitch** chunks back into full-length `[seq_len, hidden_dim]` embeddings
4. **Save** to HDF5 (one dataset per gene, gzip-compressed)

The chunking handles the fact that models have a maximum context length.
Overlapping regions are resolved by keeping the middle portion of each chunk.

In [ ]:
import h5py
import numpy as np

from foundation_models.utils.chunking import chunk_sequence, stitch_embeddings

# Configuration
CHUNK_SIZE = 8192
OVERLAP = 256
OUTPUT_DIR = Path("/tmp/fm_demo/embeddings/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hdf5_path = OUTPUT_DIR / "embeddings.h5"
hidden_dim = embedder.model.hidden_dim
model_label = f"{BACKEND}-{MODEL_SIZE}" if BACKEND == "evo2" else f"{BACKEND}-{HYENADNA_SIZE}"

t0 = time.time()

with h5py.File(hdf5_path, "w") as hf:
    hf.attrs["model"] = model_label
    hf.attrs["hidden_dim"] = hidden_dim

    for row in gene_data.iter_rows(named=True):
        gene_id = row["gene_id"]
        gene_name = row["gene_name"]
        sequence = row["sequence"]
        seq_len = len(sequence)

        print(f"\nProcessing {gene_name} ({gene_id}): {seq_len:,} bp")

        # Chunk the sequence into overlapping windows
        chunks = chunk_sequence(
            sequence=sequence,
            chunk_size=CHUNK_SIZE,
            overlap=OVERLAP,
        )
        print(f"  Chunks: {len(chunks)} (size={CHUNK_SIZE}, overlap={OVERLAP})")

        # Encode each chunk through the foundation model
        chunk_embs = []
        for i, chunk in enumerate(chunks):
            emb = embedder.encode(chunk.sequence)
            chunk_embs.append(emb)
            if (i + 1) % 5 == 0 or i == len(chunks) - 1:
                print(f"  Encoded {i + 1}/{len(chunks)} chunks")

        # Stitch chunks into full-length embeddings
        full_emb = stitch_embeddings(
            chunks=chunks,
            chunk_embeddings=chunk_embs,
            seq_len=seq_len,
            hidden_dim=hidden_dim,
        )

        hf.create_dataset(gene_id, data=full_emb, compression="gzip")
        size_mb = full_emb.nbytes / 1e6
        print(f"  Saved: {list(full_emb.shape)}, {size_mb:.1f} MB")

elapsed = time.time() - t0
print(f"\nTotal extraction time: {elapsed:.1f}s")
print(f"Output: {hdf5_path}")

## Step 5: Build Exon Labels

Binary labels (1=exon, 0=intron) at each nucleotide position, derived from
splice site annotations. These pair with the embeddings for classifier training.

The annotations file (`splice_sites_enhanced.tsv`) is generated by the data preparation pipeline.
If it doesn't exist yet, the cell below will generate it automatically.

In [ ]:
import pandas as pd

from foundation_models.utils.chunking import build_exon_labels

data_dir = Path("data/mane/GRCh38/")
splice_tsv = data_dir / "splice_sites_enhanced.tsv"

# Generate splice annotations if they don't exist yet
if not splice_tsv.exists():
    print(f"Splice annotations not found at {splice_tsv}")
    print("Generating from GTF (genome-wide, one-time operation)...\n")

    from agentic_spliceai.splice_engine.base_layer.data.preparation import (
        prepare_splice_site_annotations,
    )
    result = prepare_splice_site_annotations(
        output_dir=data_dir,
        genes=None,  # genome-wide
        build="GRCh38",
        annotation_source="mane",
    )
    if result["success"]:
        print(f"\nGenerated: {result['splice_sites_file']} ({result['n_sites']} sites)")
    else:
        print("Failed to generate splice annotations. Check GTF availability.")

labels_dict = {}

if splice_tsv.exists():
    splice_sites_df = pd.read_csv(splice_tsv, sep="\t")
    print(f"Loaded splice annotations: {len(splice_sites_df)} sites")

    for row in gene_data.iter_rows(named=True):
        gene_id = row["gene_id"]
        gene_name = row["gene_name"]
        seq_len = len(row["sequence"])
        gene_start = int(row["start"])

        labels = build_exon_labels(
            gene_id=gene_id,
            gene_start=gene_start,
            gene_sequence_length=seq_len,
            splice_sites_df=splice_sites_df,
        )
        labels_dict[gene_id] = labels
        exon_frac = labels.mean()
        print(f"  {gene_name}: {seq_len:,} bp, exon fraction = {exon_frac:.2%}")

    # Save alongside embeddings
    labels_path = hdf5_path.with_suffix(".labels.npz")
    np.savez(labels_path, **labels_dict)
    print(f"\nSaved labels: {labels_path}")
else:
    print(f"Could not generate splice annotations.")
    print("Ensure MANE GTF is available at data/mane/GRCh38/")
    print("Or use synthetic labels from 01_training_pipeline.ipynb")

## Step 6: Inspect Results

Verify the HDF5 structure and visualize the embedding space.

In [ ]:
import matplotlib.pyplot as plt

with h5py.File(hdf5_path, "r") as hf:
    print(f"HDF5: {hdf5_path}")
    print(f"  Model: {hf.attrs.get('model')}")
    print(f"  Hidden dim: {hf.attrs.get('hidden_dim')}")
    print(f"  Genes: {len(hf.keys())}")
    print()

    for gene_id in hf.keys():
        ds = hf[gene_id]
        size_mb = ds.nbytes / 1e6
        print(f"  {gene_id}: shape={ds.shape}, {size_mb:.1f} MB")

In [ ]:
# Visualize embedding norms along the gene (peaks often correlate with features)
with h5py.File(hdf5_path, "r") as hf:
    gene_ids = list(hf.keys())

    fig, axes = plt.subplots(len(gene_ids), 1, figsize=(14, 3 * len(gene_ids)))
    if len(gene_ids) == 1:
        axes = [axes]

    for ax, gene_id in zip(axes, gene_ids):
        # HDF5 random access: reads one gene without loading the entire file
        emb = hf[gene_id][:]
        norms = np.linalg.norm(emb, axis=1)

        # Downsample for plotting if gene is long
        if len(norms) > 5000:
            step = len(norms) // 5000
            plot_norms = norms[::step]
            plot_x = np.arange(0, len(norms), step)
        else:
            plot_norms = norms
            plot_x = np.arange(len(norms))

        ax.plot(plot_x, plot_norms, linewidth=0.5, color="#3572a5", alpha=0.8)
        ax.set_ylabel("L2 Norm")
        ax.set_title(f"{gene_id} — Embedding Norms ({len(norms):,} bp)")
        ax.set_xlim(0, len(norms))

        # Overlay exon labels if available
        if gene_id in labels_dict:
            lbl = labels_dict[gene_id]
            ax2 = ax.twinx()
            ax2.fill_between(range(len(lbl)), lbl, step="mid",
                             alpha=0.15, color="#e74c3c", label="Exon")
            ax2.set_ylim(-0.05, 1.1)
            ax2.set_ylabel("Exon", color="#e74c3c")
            ax2.tick_params(axis="y", labelcolor="#e74c3c")

    axes[-1].set_xlabel("Position (bp)")
    plt.tight_layout()
    plt.show()

## Next Steps

With real embeddings saved, train the ExonClassifier:

```bash
python examples/foundation_models/03_train_and_evaluate.py \
    --embeddings /tmp/fm_demo/embeddings/embeddings.h5 \
    --labels /tmp/fm_demo/embeddings/embeddings.labels.npz \
    --output /tmp/fm_demo/model/
```

Or follow the interactive training notebook: `01_training_pipeline.ipynb`
(replace the synthetic embedding step with the real HDF5 file from this notebook).

### Remote GPU Workflow

```bash
# Provision a cluster with Evo2 deps
python examples/foundation_models/ops_provision_cluster.py --model evo2

# Run extraction on the cluster
python examples/foundation_models/ops_run_pipeline.py --execute \
    --cluster fm-workspace --no-teardown \
    -- python examples/foundation_models/02_embedding_extraction.py \
         --genes BRCA1 TP53 --model evo2 --model-size 7b \
         --output /workspace/output/

# Download results
rsync -Pavz fm-workspace:/workspace/output/ ./output/embeddings/
```